In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datajoint as dj
import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

/workspace/.venv/lib/python3.12/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Access to djimaging database

## Define paths

In [3]:
username = !whoami
username = username[0]
username

'fsalomon'

In [4]:
home_directory = os.path.expanduser("~")
home_directory

'/gpfs01/euler/User/fsalomon'

In [5]:
# Path to djimaging
path_to_djimaging = f'{home_directory}/github/eulerlab/'

# Clone djimaging if you haven't downloaded it yet
# assert os.isdir(path_to_djimaging), 'Create target folder before cloning djimaging'
# !cd {path_to_djimaging} && git clone git@github.com:eulerlab/djimaging.git

# Install djimaging if not done yet
# assert os.isdir(os.path.join(path_to_djimaging, 'djimaging')), 'Create target folder before cloning djimaging'
# !cd {path_to_djimaging}/djimaging/ && sudo pip install -e .

## Prepare dj config

In [6]:
# Set config file
config_file = f'{home_directory}/GitHub/docker/dj_{username}_conf.json'
assert os.path.isfile(config_file), f'Set the path to your config file: {config_file}'

In [7]:
# Define a schema name or use the default name for your personal test schema
# It should start with ageuler and have some meaningful name after that
schema_name = f"ageuler_{username}_neuromodulation_ac"

In [8]:
# Do you want to use the RGC classifier?
# If so, make sure to add the respective tables into your schema.
use_rgc_classifier = False

if use_rgc_classifier:  # Define any existing outputfolder for the classifier to be saved
    output_folder = f'{home_directory}/datajoint/rgc_classifier'
    assert os.path.isdir(output_folder), f'Set path to output directory: {output_folder}'

In [9]:
# Load configuration for user
dj.config.load(config_file)
dj.config['schema_name'] = schema_name

if use_rgc_classifier:
    from djimaging.tables.classifier.rgc_classifier import prepare_dj_config_rgc_classifier
    prepare_dj_config_rgc_classifier(output_folder)

print("schema_name:", dj.config['schema_name'])
dj.conn()

[2026-10-08 14:06:28,949][INFO]: DataJoint is configured from /gpfs01/euler/User/fsalomon/GitHub/docker/dj_fsalomon_conf.json
[2026-10-08 14:06:29,036][INFO]: DataJoint 0.14.6 connected to fsalomon@172.25.240.200:3306


schema_name: ageuler_fsalomon_neuromodulation_ac


DataJoint connection (connected) fsalomon@172.25.240.200:3306

## Create or load schema

In [10]:
# Import your own schema, which defines the tables you will have in your database and how they are connected.
from djimaging.user.amacrine_cells_neuromodulation.schemas.amacrine_cells_neuromodulation_schema import *

In [11]:
if use_rgc_classifier:
    try:
        CelltypeAssignment()
    except Exception as e:
        import warnings
        warnings.warn("""
            If you want to include the RGC classifier in your database, you have to copy the respective tables from 
            djimaging/schemas/full_rgc_schema.py 
            to the schema you just imported (see above).
            Then restart the notebook and try again.
            """)

In [12]:
from djimaging.utils.dj_utils import activate_schema

activate_schema(schema=schema, create_schema=True, create_tables=True)
schema

Schema `ageuler_fsalomon_neuromodulation_ac`

In [13]:
target_dates = [
    datetime.date(2025, 9, 8),
    datetime.date(2025, 10, 23),
    datetime.date(2025, 11, 19)]

stimuli_chirp = [
    "gChirp",
    "lChirp"]

In [14]:
Averages() & [{"date": d} for d in target_dates] & [{"stim_name": s} for s in stimuli_chirp]

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,preprocess_id unique param set id,average array of snippet average (time),average_norm normalized array of snippet average (time),average_t0 time of the first sample of the average,average_dt time between samples of the average,triggertimes_rel array of relative triggertimes
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,1,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,2,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,3,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,4,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,5,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,6,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,7,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,8,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,9,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,10,1,=BLOB=,=BLOB=,0.0,0.002,=BLOB=


In [15]:
df_averages = pd.DataFrame((Averages & [{"date": d} for d in target_dates] & [{"stim_name": s} for s in stimuli_chirp]).fetch(as_dict=True))

In [16]:
ChirpQI & [{"date": d} for d in target_dates]

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,preprocess_id unique param set id,qidx quality index as signal to noise ratio,min_qidx minimum quality index as 1/r (r = #repetitions)
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,1,1,0.205932,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,2,1,0.26708,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,3,1,0.23731,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,4,1,0.215393,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,5,1,0.223126,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,6,1,0.194005,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,7,1,0.279805,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,8,1,0.26186,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,9,1,0.24477,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,10,1,0.207242,0.2


In [17]:
df_qi = pd.DataFrame((ChirpQI & [{"date": d} for d in target_dates]).fetch(as_dict=True))

In [18]:
keys = ["experimenter", "date", "exp_num", "raw_id", "field",
        "region", "cond1", "stim_name", "cond2", "roi_id"]

df_chirp = pd.merge(df_averages, df_qi, on=keys, how="inner", validate="one_to_one")

In [19]:
df_chirp

,experimenter,date,exp_num,raw_id,field,region,cond1,stim_name,cond2,roi_id,preprocess_id_x,average,average_norm,average_t0,average_dt,triggertimes_rel,preprocess_id_y,qidx,min_qidx
0,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,1,1,"[0.3020965666999527, 0.29009884088328863, 0.27...","[0.18146847385880732, 0.17426147704480882, 0.1...",0.0,0.002,"[0.0, 4.9972005]",1,0.205932,0.2
1,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,2,1,"[-0.5160188287740086, -0.5165614524168405, -0....","[-0.2809652603237992, -0.281260711544888, -0.2...",0.0,0.002,"[0.0, 4.9972005]",1,0.267080,0.2
2,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,3,1,"[1.1978822622336338, 1.2121204542229997, 1.216...","[0.6441296759490328, 0.651785889152415, 0.6541...",0.0,0.002,"[0.0, 4.9972005]",1,0.237310,0.2
3,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,4,1,"[-0.9645419550537577, -0.9637391988597322, -0....","[-0.5154942371550408, -0.5150652084438557, -0....",0.0,0.002,"[0.0, 4.9972005]",1,0.215393,0.2
4,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,5,1,"[1.1798392616702489, 1.170918293881814, 1.1619...","[0.6532751486949463, 0.6483356228223774, 0.643...",0.0,0.002,"[0.0, 4.9972005]",1,0.223126,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11771,Salomon,2025-11-19,2,1,XY5,RR,t,lChirp,DETANO,183,1,"[-0.0015441805374625695, 0.013436284100749773,...","[-0.0009128279740868759, 0.00794273447785921, ...",0.0,0.002,"[0.0, 4.9967995]",1,0.231606,0.2
11772,Salomon,2025-11-19,2,1,XY5,RR,t,lChirp,DETANO,184,1,"[-0.16822142745049473, -0.17530692490372193, -...","[-0.11781647602544112, -0.12277891364993539, -...",0.0,0.002,"[0.0, 4.9967995]",1,0.310748,0.2
11773,Salomon,2025-11-19,2,1,XY5,RR,t,lChirp,DETANO,185,1,"[-0.8037525242564936, -0.8069455002397501, -0....","[-0.5714345631693178, -0.5737046360849707, -0....",0.0,0.002,"[0.0, 4.9967995]",1,0.300924,0.2
11774,Salomon,2025-11-19,2,1,XY5,RR,t,lChirp,DETANO,186,1,"[-0.4612436726046635, -0.4727987720686865, -0....","[-0.3420392442403348, -0.3506080284265276, -0....",0.0,0.002,"[0.0, 4.9967995]",1,0.195492,0.2


In [20]:
df_g = df_chirp[df_chirp["stim_name"] == "gChirp"]
df_l = df_chirp[df_chirp["stim_name"] == "lChirp"]

In [21]:
import pandas as pd

keys = ["experimenter", "date", "exp_num", "raw_id", "field",
        "region", "cond1", "cond2", "roi_id"]

# Drop stim_name since it's constant within each df and would just get duplicated
g = df_g.drop(columns="stim_name")
l = df_l.drop(columns="stim_name")

merged = pd.merge(
    g, l,
    on=keys,
    how="inner",              # keep only ROIs present in both
    suffixes=("_gChirp", "_lChirp"),
    validate="one_to_one",    # raises an error if keys aren't unique per ROI
)

merged

,experimenter,date,exp_num,raw_id,field,region,cond1,cond2,roi_id,preprocess_id_x_gChirp,...,min_qidx_gChirp,preprocess_id_x_lChirp,average_lChirp,average_norm_lChirp,average_t0_lChirp,average_dt_lChirp,triggertimes_rel_lChirp,preprocess_id_y_lChirp,qidx_lChirp,min_qidx_lChirp
0,Salomon,2025-09-08,2,1,XY1,RR,v,C1,1,1,...,0.2,1,"[0.18923896339515892, 0.1863652814438447, 0.19...","[0.11934750157579792, 0.11753515408107693, 0.1...",0.0,0.002,"[0.0, 4.9926686]",1,0.327583,0.333333
1,Salomon,2025-09-08,2,1,XY1,RR,v,C1,2,1,...,0.2,1,"[-0.7860867420526079, -0.7846802397173945, -0....","[-0.46970218420618914, -0.4688617715855466, -0...",0.0,0.002,"[0.0, 4.9926686]",1,0.336615,0.333333
2,Salomon,2025-09-08,2,1,XY1,RR,v,C1,3,1,...,0.2,1,"[0.3434269042900831, 0.3476441015258344, 0.351...","[0.16435548901024438, 0.1663737336651509, 0.16...",0.0,0.002,"[0.0, 4.9926686]",1,0.331362,0.333333
3,Salomon,2025-09-08,2,1,XY1,RR,v,C1,4,1,...,0.2,1,"[-1.045836742586038, -1.036533656071143, -1.02...","[-0.5802934366788253, -0.5751315219882972, -0....",0.0,0.002,"[0.0, 4.9926686]",1,0.305237,0.333333
4,Salomon,2025-09-08,2,1,XY1,RR,v,C1,5,1,...,0.2,1,"[-0.20732950318526178, -0.19314435008213868, -...","[-0.12072813756883331, -0.11246811143189446, -...",0.0,0.002,"[0.0, 4.9926686]",1,0.340473,0.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5883,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,183,1,...,0.2,1,"[-0.0015441805374625695, 0.013436284100749773,...","[-0.0009128279740868759, 0.00794273447785921, ...",0.0,0.002,"[0.0, 4.9967995]",1,0.231606,0.200000
5884,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,184,1,...,0.2,1,"[-0.16822142745049473, -0.17530692490372193, -...","[-0.11781647602544112, -0.12277891364993539, -...",0.0,0.002,"[0.0, 4.9967995]",1,0.310748,0.200000
5885,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,185,1,...,0.2,1,"[-0.8037525242564936, -0.8069455002397501, -0....","[-0.5714345631693178, -0.5737046360849707, -0....",0.0,0.002,"[0.0, 4.9967995]",1,0.300924,0.200000
5886,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,186,1,...,0.2,1,"[-0.4612436726046635, -0.4727987720686865, -0....","[-0.3420392442403348, -0.3506080284265276, -0....",0.0,0.002,"[0.0, 4.9967995]",1,0.195492,0.200000


In [22]:
OsDsIndexes & [{"date": d} for d in target_dates]

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,preprocess_id unique param set id,ds_index direction selectivity index as resulting vector length (absolute of projection on complex exponential),ds_pvalue p-value indicating the percentile of the vector length in null distribution,pref_dir preferred direction,os_index orientation selectivity index in analogy to ds_index,os_pvalue analogous to ds_pvalue for orientation tuning,pref_or preferred orientation,on_off on off index based on time kernel,d_qi quality index for moving bar response,dir_component,time_component,time_component_dt,surrogate_v computed by projecting on time,surrogate_dsi DSI of surrogate v
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,1,1,0.119384,0.78,2.61936,0.276144,0.29,1.25174,-0.3,0.424169,=BLOB=,=BLOB=,0.128,=BLOB=,0.0997333
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,2,1,0.377681,0.0,-0.897657,0.154824,0.58,2.86767,-0.06,0.573542,=BLOB=,=BLOB=,0.128,=BLOB=,0.477601
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,3,1,0.132585,0.44,2.97996,0.4832,0.02,0.861053,-0.24,0.425207,=BLOB=,=BLOB=,0.128,=BLOB=,0.222633
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,4,1,0.227529,0.91,3.10276,0.15854,0.43,0.351112,0.12,0.508752,=BLOB=,=BLOB=,0.128,=BLOB=,0.0879941
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,5,1,0.363852,0.53,-1.02615,0.0597571,0.77,2.6067,-0.04,0.386349,=BLOB=,=BLOB=,0.128,=BLOB=,0.238921
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,6,1,0.390525,0.87,-0.438148,0.328013,0.0,1.16087,-0.4,0.430251,=BLOB=,=BLOB=,0.128,=BLOB=,0.0637796
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,7,1,0.141882,0.41,3.05762,0.410971,0.01,2.32308,-0.17,0.514681,=BLOB=,=BLOB=,0.128,=BLOB=,0.192427
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,8,1,0.312263,0.3,-0.614178,0.2013,0.74,2.49186,-0.33,0.521656,=BLOB=,=BLOB=,0.128,=BLOB=,0.241418
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,9,1,0.142486,0.25,2.28018,0.189426,0.03,0.542234,-0.1,0.40619,=BLOB=,=BLOB=,0.128,=BLOB=,0.130268
Salomon,2025-09-08,2,1,XY1,RR,v,movingbar,C1,10,1,0.379043,0.0,2.01922,0.0711011,0.57,0.750403,0.41,0.538149,=BLOB=,=BLOB=,0.128,=BLOB=,0.493194


In [23]:
df_averages_mb = pd.DataFrame((Averages & [{"date": d} for d in target_dates] & {"stim_name": "movingbar"}).fetch(as_dict=True))

In [24]:
df_osds = pd.DataFrame((OsDsIndexes & [{"date": d} for d in target_dates] & {"stim_name": "movingbar"}).fetch(as_dict=True))

In [25]:
keys = ["experimenter", "date", "exp_num", "raw_id", "field",
        "region", "cond1", "stim_name", "cond2", "roi_id"]

df_mb = pd.merge(df_averages_mb, df_osds, on=keys, how="inner", validate="one_to_one")

In [26]:
from functools import reduce
import pandas as pd

keys = ["experimenter", "date", "exp_num", "raw_id", "field",
        "region", "cond1", "cond2", "roi_id"]

dfs = {"gChirp": df_g, "lChirp": df_l, "movingbar": df_mb}

prepared = [
    df.drop(columns="stim_name").set_index(keys).add_suffix(f"_{name}")
    for name, df in dfs.items()
]

merged = reduce(lambda a, b: a.join(b, how="inner"), prepared).reset_index()

In [27]:
merged

,experimenter,date,exp_num,raw_id,field,region,cond1,cond2,roi_id,preprocess_id_x_gChirp,...,os_index_movingbar,os_pvalue_movingbar,pref_or_movingbar,on_off_movingbar,d_qi_movingbar,dir_component_movingbar,time_component_movingbar,time_component_dt_movingbar,surrogate_v_movingbar,surrogate_dsi_movingbar
0,Salomon,2025-09-08,2,1,XY1,RR,v,C1,1,1,...,0.276144,0.29,1.251740,-0.30,0.424169,"[0.2692992, 0.21163529, 1.0, 0.0, 0.5561289, 0...","[0.26338604, -0.32434568, 0.50420314, 0.173609...",0.128,"[0.40229368, 0.0, 1.0, 0.22974569, 0.2547209, ...",0.099733
1,Salomon,2025-09-08,2,1,XY1,RR,v,C1,2,1,...,0.154824,0.58,2.867670,-0.06,0.573542,"[0.57404375, 0.5936899, 0.0, 0.09891763, 0.595...","[0.23134468, 0.12591584, -0.19623627, 0.122565...",0.128,"[0.71617424, 0.42367765, 0.0, 0.07766017, 0.33...",0.477601
2,Salomon,2025-09-08,2,1,XY1,RR,v,C1,3,1,...,0.483200,0.02,0.861053,-0.24,0.425207,"[0.35994393, 0.4662192, 0.58913016, 0.14168158...","[0.47586474, -0.5384664, -0.35983962, -0.44003...",0.128,"[0.3359132, 0.3226042, 0.47080025, 0.0, 0.3162...",0.222633
3,Salomon,2025-09-08,2,1,XY1,RR,v,C1,4,1,...,0.158540,0.43,0.351112,0.12,0.508752,"[0.05255287, 1.0, 0.0, 0.9599831, 0.82622796, ...","[-0.6013447, 0.066103116, 0.07772926, 0.050788...",0.128,"[0.35936928, 0.6933557, 0.0, 1.0, 0.77427435, ...",0.087994
4,Salomon,2025-09-08,2,1,XY1,RR,v,C1,5,1,...,0.059757,0.77,2.606700,-0.04,0.386349,"[0.09850365, 0.63157904, 0.0, 0.045831554, 0.4...","[0.06198706, 0.373452, 0.6631308, -0.64670223,...",0.128,"[0.2739429, 0.9214205, 0.08431966, 0.12541033,...",0.238921
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5883,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,183,1,...,0.380884,0.52,0.697020,-0.09,0.585752,"[0.62305814, 0.83217067, 0.75547975, 0.0, 0.95...","[-0.1885956, -0.36162624, 0.16240476, -0.14663...",0.128,"[0.68252426, 0.9017159, 0.7610579, 0.0, 0.6794...",0.079812
5884,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,184,1,...,0.277532,0.29,3.039160,-0.56,0.835164,"[1.0, 0.7128995, 0.36777276, 0.7867848, 0.5617...","[0.007193401, -0.019763572, -0.13062204, -0.14...",0.128,"[0.50987834, 1.0, 0.6664306, 0.41034573, 0.532...",0.281261
5885,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,185,1,...,0.357916,0.42,2.328340,-0.46,0.821772,"[0.57470983, 0.21392569, 0.807794, 1.0, 0.1688...","[-0.08637476, 0.055225626, -0.07644735, -0.058...",0.128,"[0.6169957, 0.21917917, 0.80156857, 1.0, 0.209...",0.310006
5886,Salomon,2025-11-19,2,1,XY5,RR,t,DETANO,186,1,...,0.404686,0.04,0.370011,-0.71,0.543847,"[0.8661915, 1.0, 0.4473702, 0.32599086, 0.9086...","[-0.0846753, 0.23093115, -0.014116062, -0.1953...",0.128,"[0.99734974, 0.97553116, 0.46213415, 0.4915824...",0.080759


In [28]:
merged.to_pickle(f"../datasets/Clustering.pkl")